# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [ ]:
!pip -q install transformers sentencepiece torch


In [2]:
from transformers import pipeline, set_seed

print("Libraries loaded successfully!")


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
objc[11308]: Class AVFFrameReceiver is implemented in both /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/av/.dylibs/libavdevice.61.3.100.dylib (0x1126f03a8) and /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x700875c3a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[11308]: Class AVFAudioReceiver is implemented in both /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/av/.dylibs/libavdevice.61.3.100.dylib (0x1126f03f8) and /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-pack

Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [3]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(output[0]["generated_text"])

Device set to use mps:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"))

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [5]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

Device set to use mps:0


[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

1. **Sentiment classification:** Encoder-only
2. **Machine translation:** Encoder-decoder
3. **Open-ended text generation:** Decoder-only
4. **Named Entity Recognition:** Encoder-only
5. **Text summarization:** Encoder-decoder


## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [12]:
from transformers import pipeline

# Load the encoder-only sentiment classifier
classifier= pipeline("sentiment-analysis", model= "distilbert-base-uncased-finetuned-sst-2-english")

sentences= [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers."
]

# Run classification and print results
for sentence in sentences:
    result= classifier(sentence)[0]
    print(f"Sentence: '{sentence}'")
    print(f"Label: {result['label']} | Score: {result['score']:.4f}\n")

Device set to use mps:0


Sentence: 'The movie was excellent.'
Label: POSITIVE | Score: 0.9999

Sentence: 'The lecture was difficult to understand.'
Label: NEGATIVE | Score: 0.9997

Sentence: 'I enjoyed learning about transformers.'
Label: POSITIVE | Score: 0.9997



## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [14]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load the encoder-decoder model and tokenizer
tokenizer= AutoTokenizer.from_pretrained("google/flan-t5-small")
model= AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

paragraph= "The Machine Learning field is a field within AI that contains multiple things to make the tasks easier than before, such as summarizing, generating new cotents, and so on."

prompt= f"Summarize the following paragraph: {paragraph}"

# Convert text into tokens
inputs= tokenizer(prompt, return_tensors= "pt")
outputs= model.generate(**inputs, max_new_tokens= 30)

# Convert tokens back to text
summary= tokenizer.decode(outputs[0], skip_special_tokens= True)
print("Summary:", summary)

Summary: The machine learning field is a field that is used to learn and use the data to learn.



## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**     
A decoder-only model is suitable for a chatbot like ChatGPT because its core architecture is designed for autoregressive next-token prediction. It takes the conversation history as context and continuously calculates the probability of the following words to generate a response line by line. This unidirectional flow is optimized specifically for open-ended, continuous text generation rather than just extracting or analyzing existing text.


# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.  
When text generation utilizes sampling without a fixed random seed (set_seed), the model selects the next token based on a probability distribution rather than defaulting strictly to the single most probable word (greedy decoding). This probability-based sampling introduces randomness into the generation process, causing the model to take different creative paths and produce varied outputs from the exact same starting prompt.


In [15]:
# Load the decoder-only generator
generator= pipeline("text-generation", model= "distilgpt2")
prompt= "Natural language processing is"

print("==== Experiment: Using set_seed(42) ====")
# When set_seed is used, the model produces the exact same text every time
for run in range(1, 4):
    set_seed(42) # Locking the random seed before generation
    output= generator(
        prompt, 
        max_new_tokens= 20, 
        do_sample= True, 
        pad_token_id= 50256
    )
    print(f"Run {run}: {output[0]['generated_text']}")

print("\n==== Experiment: Removing set_seed ====")
# When we remove the seed, the output varies
for run in range(1, 4):
    output= generator(
        prompt, 
        max_new_tokens= 20, 
        do_sample= True, 
        pad_token_id= 50256
    )
    print(f"Run {run}: {output[0]['generated_text']}")

Device set to use mps:0


==== Experiment: Using set_seed(42) ====
Run 1: Natural language processing is the most common language used by professional software developers. A language processing program that can process thousands of words
Run 2: Natural language processing is the most common language used by professional software developers. A language processing program that can process thousands of words
Run 3: Natural language processing is the most common language used by professional software developers. A language processing program that can process thousands of words

==== Experiment: Removing set_seed ====
Run 1: Natural language processing is an important part of the overall English-speaking language. It is a complex learning process which can arise
Run 2: Natural language processing is a crucial part of many languages. Many languages are thought of as a function of the word order.
Run 3: Natural language processing is now the foundation of the modern language processing process. It’s a necessary part o